# Complex Systems QLoRA: training (Kaggle, one T4)

Fine-tunes the base model in `config/models.yaml` with QLoRA (nf4, r=16, alpha=32) using
`python -m app.train.qlora` from [complex-systems-lora](https://github.com/WaiteM124/complex-systems-lora).
This notebook is a thin wrapper: all logic lives in the package.

**Before Run All**
1. *Add Input*: the private dataset holding `reviewed.jsonl` (any mount path works).
2. *Settings > Accelerator*: **GPU T4 x2** (only one GPU is used).
3. *Settings > Internet*: **on** (pip and model download).
4. Optional: *Add-ons > Secrets*: `HF_TOKEN` (write access) to push the adapter to the Hub.

Keep this notebook **private**: `/kaggle/working` holds checkpoints trained on private data.

In [ ]:
import os
import subprocess
import sys
from importlib import metadata

# Use one T4 of Kaggle's pair. Set before anything imports torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["APP_ENV"] = "local"

REPO_URL = "https://github.com/WaiteM124/complex-systems-lora.git"
REPO_REF = "main"  # or a commit SHA / tag for a reproducible run
REPO_DIR = "/tmp/complex-systems-lora"


def run(*args: str) -> None:
    """Run a command in the repo and stream its output; raise if it fails."""
    proc = subprocess.Popen(
        list(args), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed: {' '.join(args)}")

In [ ]:
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], check=False)

In [ ]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REPO_REF], check=True)
commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

In [ ]:
# Pinned GPU stack from the repo's `train` extra (pyproject.toml). First try to keep
# the Kaggle image's own torch build (the constraint stops pip replacing it); if that
# torch is outside unsloth's supported range, let pip install a compatible torch.
torch_version = metadata.version("torch").split("+")[0]
with open("/tmp/constraints.txt", "w", encoding="utf-8") as handle:
    handle.write(f"torch=={torch_version}\n")
pip = [sys.executable, "-m", "pip", "install", "--quiet"]
try:
    subprocess.run([*pip, "-c", "/tmp/constraints.txt", "-e", f"{REPO_DIR}[train]"], check=True)
    print("kept Kaggle torch", torch_version)
except subprocess.CalledProcessError:
    print(f"torch {torch_version} does not fit the pins; installing a compatible torch")
    subprocess.run([*pip, "-e", f"{REPO_DIR}[train]"], check=True)
for name in (
    "torch",
    "unsloth",
    "transformers",
    "trl",
    "peft",
    "bitsandbytes",
    "accelerate",
    "datasets",
):
    print(f"{name}=={metadata.version(name)}")

In [ ]:
# Optional: add a Kaggle Secret named HF_TOKEN (Add-ons > Secrets) to push the adapter.
# The value is passed to the training process via the environment and never printed.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found: the adapter will be pushed to the Hugging Face Hub.")
except Exception:
    os.environ.pop("HF_TOKEN", None)
    print("No HF_TOKEN secret: the adapter stays in /kaggle/working/adapter.")

In [ ]:
# Finds reviewed.jsonl under /kaggle/input (any mount path), checks the schema and
# that no test-split item is in train/val, and prints the split counts.
run(sys.executable, "-m", "app.train.qlora", "--config", "config/train.yaml", "--check-data")

## Smoke run

20 steps with the full recipe, logging peak VRAM. If CUDA runs out of memory it steps down
the OOM ladder in `config/train.yaml` (batch 1 x accumulation 16, then sequence length 1536)
and records the rung that fits, so the full run starts there.

In [ ]:
run(sys.executable, "-m", "app.train.qlora", "--config", "config/train.yaml", "--smoke")

In [ ]:
import json

smoke = json.load(open("/kaggle/working/smoke_summary.json", encoding="utf-8"))
print(
    json.dumps(
        {
            k: smoke[k]
            for k in (
                "gpu",
                "rung",
                "per_device_train_batch_size",
                "gradient_accumulation_steps",
                "max_seq_length",
                "global_step",
                "train_runtime_s",
                "peak_vram_reserved_gib",
                "peak_vram_allocated_gib",
            )
        },
        indent=2,
    )
)

## Full run

3 epochs; checkpoints every `save_steps` under `/kaggle/working/checkpoints` (re-running this cell
resumes from the latest one). The newest checkpoints are scored by validation answerability F1 and
the best adapter is saved to `/kaggle/working/adapter` (and pushed to the Hub if `HF_TOKEN` is set).
Tip: *Save Version > Save & Run All* runs it in the background (sessions last up to 12 h).

In [ ]:
run(sys.executable, "-m", "app.train.qlora", "--config", "config/train.yaml")

In [ ]:
summary = json.load(open("/kaggle/working/train_summary.json", encoding="utf-8"))
keys = (
    "gpu",
    "base_model",
    "rung",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "max_seq_length",
    "n_train",
    "n_val",
    "n_test",
    "dropped_too_long",
    "global_step",
    "train_loss",
    "train_runtime_s",
    "peak_vram_reserved_gib",
    "peak_vram_allocated_gib",
    "best_checkpoint",
    "val_scores",
    "val_metrics",
    "test_split_sha256",
    "hub_pushed",
)
print(json.dumps({k: summary.get(k) for k in keys}, indent=2))